In [ ]:
# ------------------------------------------------------------------------
# a2a_agents.py  – Microsoft Agent Framework (MAF) version
# ------------------------------------------------------------------------
import abc
import asyncio
import logging
import random
from collections.abc import AsyncIterable
from typing import Any, Callable, Literal

from pydantic import BaseModel

from agent_framework import Agent, MCPStreamableHTTPTool
from agent_framework.openai import OpenAIChatCompletionClient

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)


# region Response Format
class AgentResponse(BaseModel):
    status: Literal["input_required", "completed", "error"] = "input_required"
    message: str
# endregion


class AbstractAgent(abc.ABC):
    SUPPORTED_CONTENT_TYPES = ["text", "text/plain"]

    async def __aenter__(self):
        return self

    async def __aexit__(self, exc_type, exc, tb):
        return False

    @abc.abstractmethod
    async def invoke(self, user_input: str, session_id: str) -> dict[str, Any]:
        ...


class MAFAgent(AbstractAgent):
    """Microsoft Agent Framework agent with MCP tools + automatic reconnect."""

    def __init__(
        self,
        mcp_url: str,
        title: str,
        chat_client: OpenAIChatCompletionClient,
        *,
        max_attempts: int = 3,
        base_delay: float = 0.5,
        max_delay: float = 5.0,
    ):
        self._mcp_url = mcp_url.rstrip("/")
        self._title = title
        self._chat_client = chat_client

        self._max_attempts = max_attempts
        self._base_delay = base_delay
        self._max_delay = max_delay

        self.mcp_tool: MCPStreamableHTTPTool | None = None
        self.agent: Agent | None = None
        self._sessions: dict[str, Any] = {}

    async def __aenter__(self):
        await self._open_tool_and_agent()
        return self

    async def __aexit__(self, exc_type, exc, tb):
        await self._close_everything(exc_type, exc, tb)
        return False

    async def invoke(self, user_input: str, session_id: str) -> dict[str, Any]:
        async def _call():
            session = self._ensure_session(session_id)
            response = await self.agent.run(
                user_input,
                session=session,
                tools=self.mcp_tool,
                options={"response_format": AgentResponse},
            )
            return self._get_agent_response(response)

        return await self._retry_coro("invoke", _call)

    async def stream(self, user_input: str, session_id: str) -> AsyncIterable[dict[str, Any]]:
        async def _stream_call():
            yield {
                "is_task_complete": False,
                "require_user_input": False,
                "content": "Processing the request…",
            }
            session = self._ensure_session(session_id)
            response = await self.agent.run(
                user_input,
                session=session,
                tools=self.mcp_tool,
                options={"response_format": AgentResponse},
            )
            yield self._get_agent_response(response)

        async for item in self._retry_gen("stream", _stream_call):
            yield item

    async def _retry_coro(self, op_name: str, factory: Callable[[], Any]):
        for attempt in range(1, self._max_attempts + 1):
            try:
                return await factory()
            except (ConnectionError, RuntimeError, OSError) as ex:
                await self._backoff_or_raise(op_name, attempt, ex)

    async def _retry_gen(self, op_name: str, factory: Callable[[], Any]):
        for attempt in range(1, self._max_attempts + 1):
            try:
                async for item in factory():
                    yield item
                return
            except (ConnectionError, RuntimeError, OSError) as ex:
                await self._backoff_or_raise(op_name, attempt, ex)

    async def _backoff_or_raise(self, op_name: str, attempt: int, ex: Exception):
        logger.warning("%s: connection dropped (attempt %d/%d): %s", op_name, attempt, self._max_attempts, ex)
        if attempt == self._max_attempts:
            raise
        await self._reconnect_tool()
        delay = min(self._max_delay, self._base_delay * 2 ** (attempt - 1))
        delay *= random.uniform(0.8, 1.2)
        await asyncio.sleep(delay)

    async def _reconnect_tool(self):
        logger.info("Reconnecting MCP tool for %s…", self._title)
        await self._close_everything(None, None, None)
        await self._open_tool_and_agent()

    async def _open_tool_and_agent(self):
        self.mcp_tool = MCPStreamableHTTPTool(
            name=self._title,
            url=self._mcp_url,
            description=f"{self._title} MCP tools",
        )
        await self.mcp_tool.__aenter__()

        self.agent = Agent(
            client=self._chat_client,
            name=f"{self._title}_agent",
            instructions=(
                f"You are a helpful assistant for {self._title}. "
                "Use the provided tools to answer, breaking the request down "
                "one question at a time."
            ),
        )
        await self.agent.__aenter__()
        logger.info("MCP tool connected (%s).", self._title)

    async def _close_everything(self, exc_type, exc, tb):
        self._sessions.clear()

        if self.agent:
            try:
                await self.agent.__aexit__(exc_type, exc, tb)
            except Exception as err:
                logger.debug("Agent close failed: %s", err)
            self.agent = None

        if self.mcp_tool:
            try:
                await self.mcp_tool.__aexit__(exc_type, exc, tb)
            except Exception as err:
                logger.debug("MCP tool close failed: %s", err)
            self.mcp_tool = None

    def _ensure_session(self, session_id: str):
        session = self._sessions.get(session_id)
        if session is None:
            session = self.agent.create_session()
            self._sessions[session_id] = session
        return session

    def _get_agent_response(self, response: Any) -> dict[str, Any]:
        structured = getattr(response, "value", None)
        if not isinstance(structured, AgentResponse):
            text = getattr(response, "text", None) or str(response)
            try:
                structured = AgentResponse.model_validate_json(text)
            except Exception:
                return {
                    "is_task_complete": False,
                    "require_user_input": True,
                    "content": "Unparseable response – please try again.",
                }

        mapping = {
            "input_required": {"is_task_complete": False, "require_user_input": True},
            "error": {"is_task_complete": False, "require_user_input": True},
            "completed": {"is_task_complete": True, "require_user_input": False},
        }
        meta = mapping.get(structured.status)
        return {**meta, "content": structured.message} if meta else {
            "is_task_complete": False,
            "require_user_input": True,
            "content": structured.message,
        }

In [ ]:
# ------------------------------------------------------------------------
# a2a_agent_exec.py
# ------------------------------------------------------------------------
import logging

from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events.event_queue import EventQueue
from a2a.types import (
    TaskArtifactUpdateEvent,
    TaskState,
    TaskStatus,
    TaskStatusUpdateEvent,
)
from a2a.utils import (
    new_agent_text_message,
    new_data_artifact,
    new_task,
    new_text_artifact,
)

# from a2a_agents import AbstractAgent
from typing_extensions import override


logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


class A2ALabAgentExecutor(AgentExecutor):
    """ Agent Executor """

    def __init__(self, agent: AbstractAgent):
        super().__init__()
        self.agent = agent

    # ---------------- async context-manager glue -------------
    async def __aenter__(self):
        if hasattr(self.agent, "__aenter__"):
            await self.agent.__aenter__()
        return self

    async def __aexit__(self, exc_type, exc, tb):
        if hasattr(self.agent, "__aexit__"):
            await self.agent.__aexit__(exc_type, exc, tb)
        return False
    # ---------------------------------------------------------

    async def execute(
        self,
        context: RequestContext,
        event_queue: EventQueue,
    ) -> None:
        query = context.get_user_input()
        task = context.current_task
        if not task:
            task = new_task(context.message)
            event_queue.enqueue_event(task)

        async for partial in self.agent.stream(query, task.contextId):
            require_input = partial['require_user_input']
            is_done = partial['is_task_complete']
            text_content = partial['content']

            if require_input:
                event_queue.enqueue_event(
                    TaskStatusUpdateEvent(
                        status=TaskStatus(
                            state=TaskState.input_required,
                            message=new_agent_text_message(
                                text_content,
                                task.contextId,
                                task.id,
                            ),
                        ),
                        final=True,
                        contextId=task.contextId,
                        taskId=task.id,
                    )
                )
            elif is_done:
                event_queue.enqueue_event(
                    TaskArtifactUpdateEvent(
                        append=False,
                        contextId=task.contextId,
                        taskId=task.id,
                        lastChunk=True,
                        artifact=new_text_artifact(
                            name='current_result',
                            description='Result of request to agent.',
                            text=text_content,
                        ),
                    )
                )
                event_queue.enqueue_event(
                    TaskStatusUpdateEvent(
                        status=TaskStatus(state=TaskState.completed),
                        final=True,
                        contextId=task.contextId,
                        taskId=task.id,
                    )
                )
            else:
                event_queue.enqueue_event(
                    TaskStatusUpdateEvent(
                        status=TaskStatus(
                            state=TaskState.working,
                            message=new_agent_text_message(
                                text_content,
                                task.contextId,
                                task.id,
                            ),
                        ),
                        final=False,
                        contextId=task.contextId,
                        taskId=task.id,
                    )
                )

    async def cancel(
        self, context: RequestContext, event_queue: EventQueue
    ) -> None:
        raise Exception('cancel not supported')


In [ ]:
import logging
import httpx

from starlette.applications import Starlette     # A2A wraps Starlette
from a2a.server.apps import A2AStarletteApplication
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore, InMemoryPushNotifier
from a2a.types import AgentCapabilities, AgentCard, AgentSkill

# from a2a_agents import AbstractAgent            # your own base class
# from a2a_agent_exec import A2ALabAgentExecutor
# from a2a_agents import MAFAgent

from agent_framework.openai import OpenAIChatCompletionClient
# ------------------------------------------------------------------------

log = logging.getLogger(__name__)

TITLE                   = "Weather"
MCP_URL                 = "/weather/mcp"
APIM_GATEWAY_URL        = "https://apim-ulp3pavelxqec.azure-api.net"
APIM_SUBSCRIPTION_KEY   = "**************"  # Keep it secret! Keep it safe!
OPENAI_API_VERSION      = "2024-10-21"
OPENAI_DEPLOYMENT_NAME  = "gpt-4o-mini"
ACA_URL                 = f"https://{os.environ.get('CONTAINER_APP_NAME', '')}.{os.environ.get('CONTAINER_APP_ENV_DNS_SUFFIX', '')}"
A2A_URL                 = "http://localhost:10020"

def build_app(
    *,
    host: str = "localhost",
    port: int = 10020,
) -> Starlette:
    """
    Assemble and return the fully-wired Starlette ASGI application.

    This function:
      • creates the Microsoft Agent Framework (MAF) agent
      • wraps it in an A2A executor
      • registers startup/shutdown hooks so the MCP connection is opened/closed
      • builds the A2A Starlette application and returns it
    """

    # -------- 1. Create the MAFAgent ----------------------------------
    weather_agent = MAFAgent(
        mcp_url=f"{APIM_GATEWAY_URL}{MCP_URL}",
        title=TITLE,
        chat_client=OpenAIChatCompletionClient(
            model=OPENAI_DEPLOYMENT_NAME,
            azure_endpoint=APIM_GATEWAY_URL,
            api_key=APIM_SUBSCRIPTION_KEY,
            api_version=OPENAI_API_VERSION,
        ),
    )

    # -------- 2. Wrap it in the A2A executor --------------------------
    weather_agent_exec = A2ALabAgentExecutor(agent=weather_agent)

    # -------- 3. Wire the executor into the default request handler ---
    httpx_client   = httpx.AsyncClient()
    request_handler = DefaultRequestHandler(
        agent_executor = weather_agent_exec,
        task_store     = InMemoryTaskStore(),
        push_notifier  = InMemoryPushNotifier(httpx_client),
    )

    # -------- 4. Build the A2A server via Starlette -------------------
    server = A2AStarletteApplication(
        agent_card   = _get_agent_card(host, port),
        http_handler = request_handler,
    )
    app: Starlette = server.build()

    # -------- 5. Register lifecycle hooks to open/close the agent -----
    @app.on_event("startup")
    async def _startup() -> None:
        log.info("Opening MAFAgent MCP connection …")
        await weather_agent.__aenter__()          # opens MCP tool

    @app.on_event("shutdown")
    async def _shutdown() -> None:
        log.info("Closing MAFAgent MCP connection …")
        await weather_agent.__aexit__(None, None, None)
        await httpx_client.aclose()

    return app


# ========== Helper: build the agent-card sent to A2A clients ===========
def _get_agent_card(host: str, port: int) -> AgentCard:
    capabilities = AgentCapabilities(streaming=True)

    skill_weather = AgentSkill(
        id='weather_forecast_maf',
        name='Microsoft Agent Framework Weather forecasting agent',
        description='Answers questions about the weather anywhere in the world',
        tags=['weather', 'microsoft-agent-framework'],
        examples=[
            "What's the weather like in Cairo?",
            "Is it raining today in London and Paris?",
            "What's the capital of Sweden?",
        ],
    )

    return AgentCard(
        name='MAF Weather Agent',
        description='Microsoft Agent Framework-powered weather agent',
        url=f'http://{host}:{port}/',
        version='1.0.0',
        defaultInputModes=['text'],
        defaultOutputModes=['text'],
        capabilities=capabilities,
        skills=[skill_weather],
    )

In [ ]:
# ------------------------------------------------------------------------
# run_server.py  – launch with `python run_server.py`
# ------------------------------------------------------------------------
import uvicorn, nest_asyncio

nest_asyncio.apply()

app = build_app()                  # Starlette ASGI application
uvicorn.run(app, host="0.0.0.0", port=10020)
